## Reasoning Patch

In [1]:
%load_ext autoreload
%autoreload 2

### Overview

### Set-up

In [2]:
import torch
import gc
import pandas as pd
from tqdm import tqdm

import sys
sys.path.append("src")
import _util
import _mapping
from _intervention import prepare_batch_multitoken_intervention, batch_intervene, get_attention_freeze_hooks, prepare_batch_multitoken_steering

In [3]:
_util.print_GPU_availbility()

CUDA is available: True
Available devices:
  GPU 0: NVIDIA RTX A5500
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B

In [4]:
model_type = "GPT-OSS" # GPT-OSS or R1

if model_type == "GPT-OSS":
    model, tokenizer = _util.load_OSS()
elif model_type == "R1":
    model, tokenizer = _util.load_R1()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [5]:
prompt_type = "h_pre_result" # empty or pre_result or pre_final_sum or h or h_pre_result or h_pre_final_sum
if prompt_type:
    prompt_type = "_" + prompt_type

# Load the divided prompts dataset
if 'h1' in prompt_type:
    prompts = pd.read_csv(f"data/{model_type}/h1_prompts{prompt_type[3:]}.csv")
elif 'h' in prompt_type:
    prompts = pd.read_csv(f"data/{model_type}/h_prompts{prompt_type[2:]}.csv")
else:
    prompts = pd.read_csv(f"data/{model_type}/prompts{prompt_type}.csv")
prompts["base_sum"] = prompts["base_sum"].astype('Int64')
prompts["source_sum"] = prompts["source_sum"].astype('Int64')
print(f"loaded {len(prompts)} prompts")

loaded 256 prompts


In [6]:
intervention_loc = [25] # restatement or reasoning or restatement_and_reasoning

if type(intervention_loc) == str:
    if 'h' in prompt_type:
        if model_type == "GPT-OSS":
            intervention_ids_dict = _mapping.intervene_ids_stepwise_3_digit_h
        elif model_type == "R1":
            intervention_ids_dict = _mapping.intervene_ids_R1_3_digit_h
    else:
        if model_type == "GPT-OSS":
            intervention_ids_dict = _mapping.intervene_ids_stepwise_3_digit
        elif model_type == "R1":
            intervention_ids_dict = _mapping.intervene_ids_R1_3_digit

    if intervention_loc == "restatement":
        intervention_ids = intervention_ids_dict["restatement"]
    elif intervention_loc == "reasoning":
        intervention_ids = intervention_ids_dict["reasoning"]
    elif intervention_loc == "restatement_and_reasoning":
        intervention_ids = intervention_ids_dict["restatement"] + intervention_ids_dict["reasoning"]
elif type(intervention_loc) == list:
    intervention_ids = intervention_loc

print(intervention_ids)

[25]


In [7]:
# for i, row in prompts.iterrows():
#     base_prompt = row['base_prompt']
#     source_prompt = row['source_prompt']
#     base_prompt = tokenizer(base_prompt, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
#     source_prompt = tokenizer(source_prompt, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
#     base_prompt_len = base_prompt.shape[0]
#     source_prompt_len = source_prompt.shape[0]
#     print(f"base_prompt_len: {base_prompt_len}, source_prompt_len: {source_prompt_len}")
#     print(list(enumerate(tokenizer.convert_ids_to_tokens(base_prompt))))
#     print(list(enumerate(tokenizer.convert_ids_to_tokens(source_prompt))))

# # [112, 202, 212]
# # [110, 112, 199, 202, 209, 212]
    

In [8]:
intervention_id_to_tok_pos = {
    6: 93,
    8: 111,
    10: 201,
    12: 211,
    14: 217,
    18: 229,
    19: 232,
    20: 235,
    22: 241,
    23: 244,
    25: 250,
    26: 257,
    27: 266,
}

tok_pos_list = [intervention_id_to_tok_pos[id] for id in intervention_ids]
print(tok_pos_list)


[250]


In [11]:
STEERING_LAYER = 16
STEERING_DIRECTION = "negative"
if STEERING_DIRECTION == "negative":
    coefficient = -0.5
else:
    coefficient = 0.5
# steering_vectors = torch.load("experiments/activation_intervention/steering_vectors_gpt-oss-20b.pt")
# steering_vector = steering_vectors["backtracking"]["mean"][STEERING_LAYER]
steering_vector = torch.load("experiments/activation_intervention/intervention_effect_gpt-oss-20b.pt")

## Frozen Attention Patching

In [12]:
LAYER = 0

header = list(prompts.columns) + ['intervention_ids', 'intervention_id', 'generated_text', 'factual_label_probability', 'counterfactual_label_probability']
filepath = _util.create_csv_file(f"experiments/activation_intervention/output/{model_type}/steering", f"mean_diff_{prompt_type[1:]}_{STEERING_DIRECTION}_weak_steering_final_sum.csv", header, overwrite=True)

batch_size = 24

for i in tqdm(range(0, len(prompts), batch_size)):
    torch.cuda.empty_cache()
    gc.collect()
    batch_rows = prompts.iloc[i:i+batch_size]
    base_labels_str = [str(sum) for sum in batch_rows['base_sum'].tolist()]
    base_labels = tokenizer(base_labels_str, add_special_tokens=False, return_tensors="pt")["input_ids"]
    base_labels = base_labels.squeeze(1)
    source_labels_str = [str(sum) for sum in batch_rows['source_sum'].tolist()]
    source_labels = tokenizer(source_labels_str, add_special_tokens=False, return_tensors="pt")["input_ids"]
    source_labels = source_labels.squeeze(1)

    hooks = []
    for layer in range(1):
        tokens, source_tokens, hook = prepare_batch_multitoken_intervention(model, tokenizer, layer, tok_pos_list, batch_rows['base_prompt'].tolist(), batch_rows['source_prompt'].tolist())
        hooks.append(hook)

    steering_tokens, steering_hook = prepare_batch_multitoken_steering(model, tokenizer, STEERING_LAYER, tok_pos_list, batch_rows['base_prompt'].tolist(), coefficient * steering_vector)
    hooks.append(steering_hook)
    assert torch.equal(tokens['input_ids'], steering_tokens['input_ids'])
        
    input_length = tokens["input_ids"].shape[1]

    for j in range(1):
        # attention_freeze_hooks = get_attention_freeze_hooks(model, tokens)
        with torch.no_grad():
            output = batch_intervene(model, tokens["input_ids"], hooks, attention_mask=tokens["attention_mask"]) # attention_freeze_hooks + 
        pred_toks = output.logits[:,-1,:].argmax(dim=-1)
        prob = torch.nn.functional.softmax(output.logits[:,-1,:], dim=-1)
        base_prob = prob[torch.arange(prob.shape[0]), base_labels]
        source_prob = prob[torch.arange(prob.shape[0]), source_labels]
        tokens["input_ids"] = torch.cat([tokens["input_ids"], pred_toks.unsqueeze(-1)], dim=1)
        del output
    
    for j, (_, row) in enumerate(batch_rows.iterrows()):
        generated_text = tokenizer.decode(tokens["input_ids"][j,input_length:]).replace(tokenizer.pad_token[-1], "")
        _util.write_to_csv(filepath, row.to_list() + [intervention_ids, ', '.join(str(id) for id in intervention_ids), generated_text, base_prob[j].item(), source_prob[j].item()])

    del tokens, hook
    torch.cuda.empty_cache()
    gc.collect()


  0%|                                                                             | 0/11 [00:00<?, ?it/s]

100%|████████████████████████████████████████████████████████████████████| 11/11 [02:09<00:00, 11.74s/it]
